# FPiGA Audio Hat Shared-Math Simulation Analysis

This notebook loads the binary captures emitted by the top-level GHDL simulation for `radhdl_fpiga_audio_top`. Each capture is raw little-endian signed 32-bit stereo data (`.s32le`) with interleaved left/right samples.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

notebook_dir = Path.cwd()
candidate_dirs = [
    notebook_dir.parent.parent / 'build' / 'sim' / 'audio_hat_shared_math_top',
    notebook_dir.parent / 'build' / 'sim' / 'audio_hat_shared_math_top',
    notebook_dir,
]
out_dir = next((p for p in candidate_dirs if (p / 'manifest.json').exists()), candidate_dirs[0])
manifest = json.loads((out_dir / 'manifest.json').read_text())
sample_rate = manifest['format']['sample_rate_hz']
out_dir

## Load Captures

Missing or empty captures usually indicate a failed simulation scenario or an RTL path that did not produce I2S frames during that test window.

In [ ]:
def load_s32le(path):
    raw = np.fromfile(path, dtype='<i4')
    if raw.size == 0:
        return np.empty((0, 2), dtype=np.int32)
    return raw.reshape((-1, 2))

captures = {}
for scenario in manifest['scenarios']:
    path = out_dir / scenario['file']
    captures[scenario['name']] = load_s32le(path) if path.exists() else np.empty((0, 2), dtype=np.int32)
    print(f"{scenario['name']:18s} {captures[scenario['name']].shape[0]:5d} frames  {scenario['description']}")

## Time-Domain Plots

These plots show the first frames of each scenario. ADC passthrough should visibly follow the input ramp/sine stimulus. Synth and mix captures should be nonzero and bounded.

In [ ]:
for name, data in captures.items():
    if data.size == 0:
        continue
    n = min(len(data), 256)
    t = np.arange(n) / sample_rate
    plt.figure(figsize=(10, 3))
    plt.plot(t, data[:n, 0], label='left')
    plt.plot(t, data[:n, 1], label='right', alpha=0.75)
    plt.title(name)
    plt.xlabel('time (s)')
    plt.ylabel('sample')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Frequency-Domain Spot Check

The FFT is a quick sanity check for oscillator and poly-voice output. The exact peak can be affected by short capture windows and startup transients, but the generated scenarios should show clear spectral energy.

In [ ]:
for name in ['dac_four_osc', 'dac_poly_voice', 'dac_mix']:
    data = captures.get(name, np.empty((0, 2), dtype=np.int32))
    if len(data) < 16:
        continue
    x = data[:, 0].astype(np.float64)
    x = x - x.mean()
    window = np.hanning(len(x))
    spectrum = np.abs(np.fft.rfft(x * window))
    freqs = np.fft.rfftfreq(len(x), d=1/sample_rate)
    plt.figure(figsize=(10, 3))
    plt.plot(freqs, spectrum)
    plt.xlim(0, min(5000, sample_rate / 2))
    plt.title(f'{name} FFT')
    plt.xlabel('frequency (Hz)')
    plt.ylabel('magnitude')
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
    peak = freqs[int(np.argmax(spectrum[1:]) + 1)] if len(spectrum) > 1 else 0
    print(f'{name}: strongest non-DC bin near {peak:.1f} Hz')